# Phase 3 (Transcription) on Colab T4

Runs `src/transcription/transcribe.py` (faster-whisper `large-v3`) against a T4 GPU instead of the local T1000 (4GB VRAM) box.

**Before running:**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload `data/raw/audio/` (and ideally `data/raw/metadata/`, for the duration sum used in the RTF extrapolation) to a folder in your Google Drive.
3. Edit `DRIVE_AUDIO_DIR` and `DRIVE_TRANSCRIPTS_DIR` in the **Configure paths** cell below to match where you put it.

Output lands directly in the Drive-mounted transcripts folder, so there's no separate download step -- it's already in the same place you'd sync back to local/T1000 from.

## 1. Confirm the GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

name, memory.total [MiB], memory.used [MiB], utilization.gpu [%]
Tesla T4, 15360 MiB, 0 MiB, 0 %


## 2. Clone the repo

In [2]:
!git clone https://github.com/DAG-21/PureBillion-Cloner.git
%cd PureBillion-Cloner
!git log --oneline -5

Cloning into 'PureBillion-Cloner'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 112 (delta 34), reused 96 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 48.58 KiB | 4.86 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/PureBillion-Cloner
ebb2a4c (HEAD -> main, origin/main, origin/HEAD) Add Colab T4 notebook for Phase 3 transcription
51182a7 Set up GPU machine for Phase 3, tune transcription config for T1000 VRAM
301c56c Note Phase 3 push status in PROJECT_UPDATES.md
4ffaff9 Scaffold Phase 3 transcription stage (faster-whisper)
744385d Update PROJECT_UPDATES.md with commit, branch rename, and data status


## 3. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. Configure paths

Edit these two to match where you uploaded/want the data in your Drive, then run the cell.

In [4]:
# EDIT THESE to match your Drive layout
DRIVE_AUDIO_DIR = "/content/drive/MyDrive/Jio Institute/Live Project/audio"
DRIVE_METADATA_DIR = "/content/drive/MyDrive/Jio Institute/Live Project/metadata"  # used for RTF extrapolation only
DRIVE_TRANSCRIPTS_DIR = "/content/drive/MyDrive/Jio Institute/Live Project"

import os
assert os.path.isdir(DRIVE_AUDIO_DIR), f"Not found: {DRIVE_AUDIO_DIR} -- upload your audio there first"
os.makedirs(DRIVE_TRANSCRIPTS_DIR, exist_ok=True)
print("Audio files found:", len(os.listdir(DRIVE_AUDIO_DIR)))

Audio files found: 199


## 5. Install dependencies

Only what Phase 3 needs -- same reasoning as on the T1000 box: don't install the full `requirements.txt` (it pulls in Linux-only `vllm` and deps for stages not being run here).

In [5]:
!pip install -q faster-whisper pyyaml tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 72.3 MB/s eta 0:00:00


In [6]:
import ctranslate2
print("CUDA devices visible to ctranslate2:", ctranslate2.get_cuda_device_count())

CUDA devices visible to ctranslate2: 1


## 6. Dry run

Confirms the file list without loading the model or touching the GPU -- same check that was done on the Dell laptop before the T1000 run.

In [7]:
!python -m src.transcription.transcribe \
  --input-dir "{DRIVE_AUDIO_DIR}" \
  --output-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --dry-run

2026-08-05 12:17:58 | INFO     | __main__ | Starting transcription from: /content/drive/MyDrive/Jio Institute/Live Project/audio
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | Found 199 audio file(s) in /content/drive/MyDrive/Jio Institute/Live Project/audio
Audio files:   0% 0/199 [00:00<?, ?file/s]2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | [dry-run] Would transcribe: 0KezzzN3-4s (0KezzzN3-4s.m4a)
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | [dry-run] Would transcribe: 0LmzHByKvjE (0LmzHByKvjE.webm)
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | [dry-run] Would transcribe: 0hSCclbhNN4 (0hSCclbhNN4.webm)
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | [dry-run] Would transcribe: 1XCObQjSHIs (1XCObQjSHIs.webm)
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber | [dry-run] Would transcribe: 1dwEdhM0YsM (1dwEdhM0YsM.webm)
2026-08-05 12:17:58 | INFO     | src.transcription.transcriber

## 7. Single-file RTF benchmark

Transcribes one real file and times it, mirroring the planned T1000 benchmark step -- gives an apples-to-apples real-time-factor (RTF) number to compare T4 vs. T1000 before committing to a full run on either.

T4 has 16GB VRAM (vs. the T1000's 4GB), so this uses plain `float16` instead of the T1000's VRAM-constrained `int8_float16`.

In [8]:
import glob, os, shutil, tempfile, time

audio_files = sorted(glob.glob(os.path.join(DRIVE_AUDIO_DIR, "*")))
assert audio_files, "No audio files found in DRIVE_AUDIO_DIR"
sample_file = audio_files[0]
print("Benchmarking on:", sample_file)

# Run the benchmark against a scratch dir so it doesn't affect the real
# transcription_history.csv / skip-if-exists logic for the full run below.
bench_input_dir = tempfile.mkdtemp()
bench_output_dir = tempfile.mkdtemp()
shutil.copy(sample_file, bench_input_dir)

Benchmarking on: /content/drive/MyDrive/Jio Institute/Live Project/audio/0KezzzN3-4s.m4a


'/tmp/tmp7d3m2_sb/0KezzzN3-4s.m4a'

In [9]:
start = time.time()
!python -m src.transcription.transcribe \
  --input-dir "{bench_input_dir}" \
  --output-dir "{bench_output_dir}" \
  --history-file "{bench_output_dir}/bench_history.csv" \
  --device cuda \
  --compute-type float16
elapsed = time.time() - start
print(f"Wall clock: {elapsed:.1f}s")

2026-08-05 12:24:09 | INFO     | __main__ | Starting transcription from: /tmp/tmp7d3m2_sb
2026-08-05 12:24:09 | INFO     | src.transcription.transcriber | Found 1 audio file(s) in /tmp/tmp7d3m2_sb
Audio files:   0% 0/1 [00:00<?, ?file/s]2026-08-05 12:24:13 | INFO     | src.transcription.transcriber | Loading faster-whisper model 'large-v3' (device=cuda, compute_type=float16)...
2026-08-05 12:24:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-05 12:24:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-large-v3/revision/main "HTTP/1.1 200 OK"
2026-08-05 12:24:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-large-v3/tree/edaa852ec7e145841d8ffdb056a99866b5f0a478?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-05 12:24:14 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/Systran/faster-whisper-large-v3/resolve/ed

In [10]:
import json

transcript_path = glob.glob(os.path.join(bench_output_dir, "*.json"))[0]
with open(transcript_path) as f:
    transcript = json.load(f)

audio_duration = transcript["duration"]
rtf = elapsed / audio_duration
print(f"Audio duration: {audio_duration:.1f}s")
print(f"Transcription wall clock: {elapsed:.1f}s")
print(f"RTF (wall_clock / audio_duration): {rtf:.3f}")
print(f"-> roughly {1/rtf:.1f}x real-time on this T4")

Audio duration: 283.0s
Transcription wall clock: 64.2s
RTF (wall_clock / audio_duration): 0.227
-> roughly 4.4x real-time on this T4


In [11]:
# Extrapolate to the full corpus, if metadata is available
if os.path.isdir(DRIVE_METADATA_DIR):
    total_duration = 0.0
    for meta_file in glob.glob(os.path.join(DRIVE_METADATA_DIR, "*.json")):
        with open(meta_file) as f:
            meta = json.load(f)
        total_duration += meta.get("duration", 0)
    est_seconds = total_duration * rtf
    print(f"Total corpus duration: {total_duration/3600:.1f} hours")
    print(f"Estimated full-batch transcription time on this T4: {est_seconds/3600:.1f} hours")
else:
    print(f"DRIVE_METADATA_DIR not found ({DRIVE_METADATA_DIR}) -- skipping full-corpus estimate.")

Total corpus duration: 36.6 hours
Estimated full-batch transcription time on this T4: 8.3 hours


## 8. Full batch run

Only run this once the estimate above looks reasonable for a single Colab session (free tier disconnects on ~90 min idle and caps sessions around ~12h -- if the estimate exceeds that, either upgrade to Colab Pro or split the batch across multiple sessions, since the pipeline already skips already-transcribed IDs on rerun).

Writes straight into the Drive-mounted `DRIVE_TRANSCRIPTS_DIR`, so results persist even if the Colab runtime is later recycled.

In [ ]:
!python -m src.transcription.transcribe \
  --input-dir "{DRIVE_AUDIO_DIR}" \
  --output-dir "{DRIVE_TRANSCRIPTS_DIR}" \
  --device cuda \
  --compute-type float16

2026-08-05 12:39:08 | INFO     | __main__ | Starting transcription from: /content/drive/MyDrive/Jio Institute/Live Project/audio
2026-08-05 12:39:08 | INFO     | src.transcription.transcriber | Found 199 audio file(s) in /content/drive/MyDrive/Jio Institute/Live Project/audio
Audio files:   0% 0/199 [00:00<?, ?file/s]2026-08-05 12:39:14 | INFO     | src.transcription.transcriber | Loading faster-whisper model 'large-v3' (device=cuda, compute_type=float16)...
2026-08-05 12:39:14 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-large-v3/revision/main "HTTP/1.1 200 OK"
2026-08-05 12:39:22 | INFO     | faster_whisper | Processing audio with duration 04:43.005
2026-08-05 12:39:23 | INFO     | faster_whisper | VAD filter removed 00:29.853 of audio
2026-08-05 12:39:47 | INFO     | src.transcription.transcriber | Transcribed 0KezzzN3-4s.m4a -> /content/drive/MyDrive/Jio Institute/Live Project/0KezzzN3-4s.json
Audio files:   1% 1/199 [00:38<2:05:54

## 9. Sanity check the output

In [ ]:
import glob
n = len(glob.glob(os.path.join(DRIVE_TRANSCRIPTS_DIR, "*.json")))
print(f"{n} transcript JSON files in {DRIVE_TRANSCRIPTS_DIR}")
!head -c 500 "{glob.glob(os.path.join(DRIVE_TRANSCRIPTS_DIR, '*.json'))[0]}"

## Next steps

- Sync `DRIVE_TRANSCRIPTS_DIR` back down to local (or to the T1000 machine) so Phase 4 (diarization) has the transcripts to work from.
- `configs/transcription.yaml` in this repo clone is still set for the T1000 (`int8_float16`). This notebook overrides `--device`/`--compute-type` on the command line instead of editing the file, so nothing here needs committing back -- but note it if you ever want the config itself to default to T4 settings.
- Update `PROJECT_UPDATES.md` with the real RTF/timing numbers from step 7 once you have them.